# Why high-frequency covariance data must be cleaned before it is synchronized

## A WRDS/TAQ data-cleaning case study for `covharness`

This notebook documents market data filtering for the covariance-forecasting benchmark. It shows **why the raw trade-and-quote feed cannot be treated as if it were already a clean price process**.

We work with the WRDS millisecond TAQ sample for **13 February 2009** and use five liquid U.S. equities:

- IBM
- AAPL
- MSFT
- JPM
- XOM

The notebook deliberately proceeds in two stages:

1. construct a naive five-minute panel from raw NBBO midquotes;
2. show how a single bad quote can create an economically absurd return and contaminate a realized covariance matrix.

We then adopt a literature-based quote-cleaning procedure before synchronization.

### Main empirical lesson

Previous-tick synchronization can be implemented perfectly and still produce a bad covariance matrix if the observation being carried forward is itself erroneous. Therefore the correct measurement pipeline is

$$
\text{raw TAQ}
\rightarrow
\textbf{clean quotes}
\rightarrow
\text{midquotes}
\rightarrow
\text{previous-tick synchronization}
\rightarrow
\text{intraday returns}
\rightarrow
\text{realized covariance}.
$$

This notebook is intended to remain in the repository as a concrete example of why the cleaning stage is part of the econometric measurement design rather than a cosmetic preprocessing step.


## 1. Methodological source and an important adaptation

The quote-cleaning rules used below are adapted from:

> **Barndorff-Nielsen, O. E., Hansen, P. R., Lunde, A., and Shephard, N. (2011).  
> “Multivariate realised kernels: consistent positive semi-definite estimators of the covariation of equity prices with noise and non-synchronous trading.”  
> *Journal of Econometrics*, 162(2), 149–169.**

Section 5.1 of that paper briefly reviews the cleaning procedure the authors use and states that it follows the more detailed procedure in:

> **Barndorff-Nielsen, O. E., Hansen, P. R., Lunde, A., and Shephard, N. (2009).  
> “Realised kernels in practice: trades and quotes.”  
> *The Econometrics Journal*, 12(3), C1–C32.**

For quote data, the four rules are:

| Rule | Cleaning operation | Why it matters |
|---|---|---|
| **Q1** | If several quotes have the same timestamp, replace them by one observation using the median bid and median ask. | Prevents a burst of simultaneous messages from being treated as distinct price states. |
| **Q2** | Delete observations with a negative spread. | A negative spread means bid \(>\) ask, i.e. a crossed quote. |
| **Q3** | Delete observations whose spread exceeds 10 times that day's median spread. | Removes extremely wide quote states that are unlikely to represent the economically relevant price. |
| **Q4** | Delete a midquote if it deviates by more than 10 mean absolute deviations from a centered median computed from 50 surrounding observations, excluding the observation being tested. | Detects an abnormal price level even when the quoted spread itself is narrow. |

The paper also applies general filters such as restricting observations to regular exchange hours and deleting zero prices.

### Our adaptation

The paper's empirical implementation retains quotes from a selected exchange. In this notebook we instead use WRDS's **consolidated NBBO fields** (`best_bid`, `best_ask`). NBBO is the National Best Bid and Offer across venues. Consequently, this notebook should be described as **adapting** the Q1–Q4 procedure rather than exactly reproducing every exchange-selection decision in the paper.

For five-minute realized covariances, Barndorff-Nielsen et al. (2011) use calendar-time returns aligned by the **previous-tick** approach. That is also the synchronization mechanism demonstrated here.


## 2. Connect to WRDS

Authentication should be handled outside the notebook, e.g. with `~/.pgpass`. **Never place a WRDS password in this notebook or in the Git repository.**


In [1]:
import wrds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

db = wrds.Connection()


ModuleNotFoundError: No module named 'matplotlib'

## 3. What is actually inside the TAQ sample?

Three tables are useful for understanding the raw feed:

- `ctm_20090213`: consolidated millisecond **trades**
- `cqm_20090213`: consolidated millisecond **quote messages**
- `nbbom_20090213`: quote records carrying the **National Best Bid and Offer**

In our initial inspection WRDS reported approximately:

- **35,386,928** rows in the trade table,
- **524,966,304** rows in the quote-message table,
- **161,567,648** rows in the NBBO table.

Even a single sample day therefore contains hundreds of millions of quote updates. This is why queries should be filtered in SQL before data are transferred into Python.

The key NBBO variables for this notebook are

$$
\text{midquote}_t
=
\frac{\text{best bid}_t+\text{best ask}_t}{2},
\qquad
\text{spread}_t
=
\text{best ask}_t-\text{best bid}_t.
$$

`time_m` is seconds after midnight with millisecond precision. For example,

$$
34200.707
$$

corresponds to **09:30:00.707**, because 09:30 is 34,200 seconds after midnight.


In [ ]:
# Optional schema check. This reads metadata only; it does not download the full tables.
for table in ["ctm_20090213", "cqm_20090213", "nbbom_20090213"]:
    print(f"\n--- {table} ---")
    display(
        db.describe_table(
            library="taqmsamp_all",
            table=table
        )
    )


## 4. A five-minute IBM window: raw NBBO behavior

We first inspect IBM from 09:30 to 09:35. The goal is to see what the raw quote stream looks like before any five-minute sampling is imposed.

The original pilot produced:

- **2,321** NBBO updates in only five minutes;
- **8 crossed quotes** (negative spreads);
- **25 locked quotes** (zero spreads);
- a median spread of approximately **\$0.03**.

The first seconds after the opening bell displayed much wider spreads than the median. This is already evidence that high-frequency observations cannot be treated as homogeneous measurements.


In [ ]:
nbbo_ibm = db.raw_sql(
    '''
    SELECT
        date,
        time_m,
        sym_root,
        best_bid,
        best_ask,
        best_bidsiz,
        best_asksiz,
        best_bidex,
        best_askex,
        nbbo_qu_cond,
        qu_seqnum
    FROM taqmsamp_all.nbbom_20090213
    WHERE sym_root = 'IBM'
      AND time_m >= 34200
      AND time_m < 34500
      AND best_bid > 0
      AND best_ask > 0
    ORDER BY time_m, qu_seqnum
    '''
)

nbbo_ibm["timestamp"] = (
    pd.to_datetime(nbbo_ibm["date"])
    + pd.to_timedelta(nbbo_ibm["time_m"], unit="s")
)
nbbo_ibm["midquote"] = (nbbo_ibm["best_bid"] + nbbo_ibm["best_ask"]) / 2
nbbo_ibm["spread"] = nbbo_ibm["best_ask"] - nbbo_ibm["best_bid"]

display(
    nbbo_ibm[
        [
            "timestamp",
            "best_bid",
            "best_ask",
            "midquote",
            "spread",
            "best_bidex",
            "best_askex",
        ]
    ].head(20)
)

print("Rows:", len(nbbo_ibm))
print("Crossed quotes:", (nbbo_ibm["spread"] < 0).sum())
print("Locked quotes:", (nbbo_ibm["spread"] == 0).sum())
display(nbbo_ibm["spread"].describe())


### Crossed and locked quotes

A **crossed quote** satisfies

$$
\text{best bid}>\text{best ask},
$$

so the spread is negative. This is not a sensible quote state on which to base a midpoint and is removed by Q2.

A **locked quote** satisfies

$$
\text{best bid}=\text{best ask}.
$$

Its spread is zero rather than negative; it is therefore not removed solely for being locked.

The distinction matters because an indiscriminate filter would throw away valid zero-spread observations together with genuinely inconsistent crossed observations.


## 5. From irregular NBBO events to a five-minute IBM return series

The market does not provide a quote at exactly every five-minute clock time. We therefore sample the latest valid observation at or before each grid point:

$$
P^{PT}(t)
=
P\!\left(\max\{\tau:\tau\le t\}\right).
$$

This is previous-tick sampling.

For this sample, the first regular-session IBM quote arrives at approximately 09:30:00.868. A strict previous-tick lookup at exactly 09:30:00 therefore has no regular-session observation available and returns `NaN`.

For this pedagogical notebook we leave that boundary issue visible and drop the missing 09:30 price. **The final production convention for the market-open boundary should be specified explicitly rather than silently filled with a pre-market observation.**


In [ ]:
nbbo_day = db.raw_sql(
    '''
    SELECT
        date,
        time_m,
        sym_root,
        best_bid,
        best_ask,
        qu_seqnum
    FROM taqmsamp_all.nbbom_20090213
    WHERE sym_root = 'IBM'
      AND time_m >= 34200
      AND time_m <= 57600
      AND best_bid > 0
      AND best_ask > 0
    ORDER BY time_m, qu_seqnum
    '''
)

nbbo_day["timestamp"] = (
    pd.to_datetime(nbbo_day["date"])
    + pd.to_timedelta(nbbo_day["time_m"], unit="s")
)
nbbo_day["spread"] = nbbo_day["best_ask"] - nbbo_day["best_bid"]

# Minimal consistency filter for this first synchronization demonstration.
nbbo_day = nbbo_day.loc[nbbo_day["spread"] >= 0].copy()
nbbo_day["midquote"] = (nbbo_day["best_bid"] + nbbo_day["best_ask"]) / 2

grid = pd.date_range(
    "2009-02-13 09:30:00",
    "2009-02-13 16:00:00",
    freq="5min",
)

ibm_5min = pd.merge_asof(
    pd.DataFrame({"timestamp": grid}),
    nbbo_day[["timestamp", "midquote"]].sort_values("timestamp"),
    on="timestamp",
    direction="backward",
)

print("Missing values before the boundary decision:")
display(ibm_5min.isna().sum())

ibm_5min = ibm_5min.dropna().reset_index(drop=True)
ibm_5min["log_price"] = np.log(ibm_5min["midquote"])
ibm_5min["return_5min"] = ibm_5min["log_price"].diff()

display(ibm_5min.head(10))


The pilot produced an entirely plausible IBM series. For example, the 09:35 and 09:40 midquotes were approximately

$$
94.635 \quad\text{and}\quad 94.000,
$$

so the corresponding log return was

$$
\log(94.000)-\log(94.635)\approx -0.006733,
$$

or roughly \(-0.67\%\).

At this stage the synchronization logic appeared to be working. The important failure emerged only when we moved to several assets.


## 6. The naive five-stock panel — and the failure that motivated this notebook

We next synchronized IBM, AAPL, MSFT, JPM and XOM on the same five-minute grid.

The following cell intentionally uses only the basic session/positive-price restrictions plus Q2. It is a **diagnostic naive pipeline**, not the final cleaning procedure.


In [ ]:
symbols = ["IBM", "AAPL", "MSFT", "JPM", "XOM"]
symbols_sql = "', '".join(symbols)

nbbo_multi_raw = db.raw_sql(
    f'''
    SELECT
        date,
        time_m,
        sym_root,
        best_bid,
        best_ask,
        qu_seqnum
    FROM taqmsamp_all.nbbom_20090213
    WHERE sym_root IN ('{symbols_sql}')
      AND time_m >= 34200
      AND time_m <= 57600
      AND best_bid > 0
      AND best_ask > 0
    ORDER BY sym_root, time_m, qu_seqnum
    '''
)

nbbo_multi_raw["timestamp"] = (
    pd.to_datetime(nbbo_multi_raw["date"])
    + pd.to_timedelta(nbbo_multi_raw["time_m"], unit="s")
)
nbbo_multi_raw["spread"] = (
    nbbo_multi_raw["best_ask"] - nbbo_multi_raw["best_bid"]
)

# Q2 only for the naive panel.
nbbo_multi_naive = nbbo_multi_raw.loc[
    nbbo_multi_raw["spread"] >= 0
].copy()

nbbo_multi_naive["midquote"] = (
    nbbo_multi_naive["best_bid"] + nbbo_multi_naive["best_ask"]
) / 2

def previous_tick_panel(quotes, symbols, grid):
    sampled = []

    for symbol in symbols:
        events = (
            quotes.loc[
                quotes["sym_root"] == symbol,
                ["timestamp", "midquote"],
            ]
            .sort_values("timestamp")
        )

        tmp = pd.merge_asof(
            pd.DataFrame({"timestamp": grid}),
            events,
            on="timestamp",
            direction="backward",
        )
        tmp["symbol"] = symbol
        sampled.append(tmp)

    long = pd.concat(sampled, ignore_index=True)
    return long.pivot(
        index="timestamp",
        columns="symbol",
        values="midquote",
    )

prices_5min_naive = previous_tick_panel(
    nbbo_multi_naive,
    symbols,
    grid,
)

display(prices_5min_naive.head(10))
print("\nMissing values by stock:")
display(prices_5min_naive.isna().sum())


### The JPM anomaly

The synchronization technically succeeded: each of the five stocks had only the expected missing observation at 09:30.

But the resulting JPM series contained values such as:

| Time | Naive JPM midquote |
|---|---:|
| 09:40 | 25.055 |
| **09:45** | **22.585** |
| 09:50 | 25.185 |
| 10:00 | 25.255 |
| **10:05** | **22.620** |
| 10:10 | 25.395 |

If interpreted literally, JPM would have fallen by roughly ten percent in five minutes and almost immediately reversed — twice in less than half an hour.

That is the key diagnostic event in this notebook.

A covariance estimator does not know that such a return is implausible. If we compute

$$
\widehat{\Sigma}_d
=
\sum_j r_jr_j',
$$

an artificial ten-percent return is squared on the diagonal and multiplied by every contemporaneous asset return off the diagonal. One bad price can therefore distort **both variance and covariance estimates**.

The synchronization algorithm did not fail. It faithfully carried forward the last quote it had been given. **The input quote was the problem.**


## 7. Verify that the JPM move was a quote-data problem, not a market move

To diagnose the problem, we inspect the raw JPM NBBO stream and the trade tape around 09:45.

The actual pilot showed trades around approximately **\$24.93–\$24.96** in this window, while the unfiltered quote stream temporarily carried best quotes around the low **\$22s**. The trade tape therefore provided a useful independent sanity check: the naive five-minute midquote was not describing a genuine ten-percent market move.


In [ ]:
def inspect_jpm_window(start_second, end_second):
    quotes = db.raw_sql(
        f'''
        SELECT
            date,
            time_m,
            sym_root,
            bid,
            ask,
            bidex,
            askex,
            qu_cond,
            qu_cancel,
            natbbo_ind,
            nbbo_qu_cond,
            best_bid,
            best_ask,
            best_bidex,
            best_askex,
            best_bidsiz,
            best_asksiz,
            qu_seqnum
        FROM taqmsamp_all.nbbom_20090213
        WHERE sym_root = 'JPM'
          AND time_m >= {start_second}
          AND time_m <= {end_second}
        ORDER BY time_m, qu_seqnum
        '''
    )

    trades = db.raw_sql(
        f'''
        SELECT
            time_m,
            ex,
            price,
            size,
            tr_scond,
            tr_corr,
            tr_seqnum
        FROM taqmsamp_all.ctm_20090213
        WHERE sym_root = 'JPM'
          AND time_m >= {start_second}
          AND time_m <= {end_second}
          AND price > 0
          AND size > 0
        ORDER BY time_m, tr_seqnum
        '''
    )

    return quotes, trades


jpm_quotes_0945, jpm_trades_0945 = inspect_jpm_window(35090, 35110)

print("Quote rows:", len(jpm_quotes_0945))
print("Trade rows:", len(jpm_trades_0945))

print("\nQuotes immediately before the 09:45 grid point:")
display(
    jpm_quotes_0945.loc[
        jpm_quotes_0945["time_m"] <= 35100,
        [
            "time_m",
            "best_bid",
            "best_ask",
            "best_bidex",
            "best_askex",
            "qu_seqnum",
        ],
    ].tail(15)
)

print("\nTrade-price summary in the same 20-second window:")
display(jpm_trades_0945["price"].describe())

print("\nRepresentative trades:")
display(jpm_trades_0945.head(15))


## 8. Adopt the literature-based Q1–Q4 quote-cleaning pipeline

The diagnostic above motivates the formal cleaning stage. The code below applies Q1–Q4 **before** any five-minute synchronization.

Two details are worth emphasizing.

### Q3 and Q4 solve different problems

Q3 asks whether the **spread** is abnormal. A quote such as

$$
\text{bid}=22.00,\qquad \text{ask}=25.00
$$

has an enormous spread and is easy to flag.

But a bad quote could instead be

$$
\text{bid}=22.57,\qquad \text{ask}=22.59.
$$

Its spread is only \$0.02, so Q3 would not identify it. If surrounding JPM midquotes are near \$25, however, Q4 recognizes that the **price level itself** is locally abnormal.

### Q4 uses mean absolute deviation, not “MAD” in the common median-absolute-deviation sense

For observation \(i\), let the 50 neighboring midquotes consist of 25 observations before and 25 after \(i\), excluding \(i\). Let

$$
\widetilde m_i
=
\operatorname{median}(\mathcal N_i).
$$

The local mean absolute deviation is

$$
D_i
=
\frac{1}{50}
\sum_{j\in\mathcal N_i}
|m_j-\widetilde m_i|.
$$

Q4 deletes \(m_i\) when

$$
|m_i-\widetilde m_i|>10D_i.
$$

The paper specifies a centered neighborhood of 50 observations excluding the observation under consideration. It does not spell out the edge-window convention in the brief review. For this pilot, the first and last 25 observations of each stock-day are left untouched because they do not have a complete centered 50-neighbor window. That convention should remain explicit in the production implementation.


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view


def q4_keep_mask(midquotes, half_window=25, threshold=10.0):
    '''
    Local midpoint-outlier filter adapted from Q4 in
    Barndorff-Nielsen, Hansen, Lunde & Shephard (2011).

    For each observation with a full centered window:
      * take 25 observations before and 25 after;
      * exclude the observation being tested;
      * compute the median of those 50 neighbors;
      * compute their mean absolute deviation from that median;
      * remove the center if its deviation exceeds
        `threshold` times that mean absolute deviation.

    Edge convention for this pilot:
    the first/last `half_window` observations are retained
    because a complete centered neighborhood is unavailable.
    '''
    x = np.asarray(midquotes, dtype=float)
    n = len(x)
    keep = np.ones(n, dtype=bool)

    width = 2 * half_window + 1
    if n < width:
        return keep

    windows = sliding_window_view(x, width)

    neighbors = np.concatenate(
        [
            windows[:, :half_window],
            windows[:, half_window + 1 :],
        ],
        axis=1,
    )

    center = windows[:, half_window]
    local_median = np.median(neighbors, axis=1)
    mean_abs_dev = np.mean(
        np.abs(neighbors - local_median[:, None]),
        axis=1,
    )
    deviation = np.abs(center - local_median)

    # If all 50 neighbors are identical, any different center is an outlier.
    bad = np.where(
        mean_abs_dev == 0,
        deviation > 0,
        deviation > threshold * mean_abs_dev,
    )

    keep[half_window:-half_window] = ~bad
    return keep


def clean_nbbo_quotes(
    raw_quotes,
    q3_multiple=10.0,
    q4_half_window=25,
    q4_threshold=10.0,
):
    '''
    Apply the Q1-Q4 quote-cleaning sequence to consolidated NBBO fields.

    P1/P2 (regular trading hours and positive bid/ask) are assumed to
    have already been imposed by the SQL extraction.

    This is an adaptation of the paper's procedure because the paper
    works with selected exchange quotes whereas this notebook uses
    consolidated NBBO fields.
    '''
    q = raw_quotes.copy()

    stage_counts = [("input_after_P1_P2", len(q))]

    # Q1: same stock-day-timestamp -> median bid and median ask.
    q = (
        q.groupby(
            ["date", "sym_root", "time_m"],
            as_index=False,
        )
        .agg(
            best_bid=("best_bid", "median"),
            best_ask=("best_ask", "median"),
        )
        .sort_values(["date", "sym_root", "time_m"])
        .reset_index(drop=True)
    )
    stage_counts.append(("after_Q1_same_timestamp_median", len(q)))

    q["spread"] = q["best_ask"] - q["best_bid"]

    # Q2: remove crossed quotes.
    q = q.loc[q["spread"] >= 0].copy()
    stage_counts.append(("after_Q2_nonnegative_spread", len(q)))

    # Q3: remove spreads > 10 times stock-day median spread.
    median_spread = (
        q.groupby(["date", "sym_root"])["spread"]
        .transform("median")
    )
    q = q.loc[
        q["spread"] <= q3_multiple * median_spread
    ].copy()
    stage_counts.append(("after_Q3_spread_filter", len(q)))

    q["midquote"] = (q["best_bid"] + q["best_ask"]) / 2

    # Q4: local midquote-level outlier filter, separately by stock-day.
    pieces = []
    q4_removed = 0

    for (_, _), g in q.groupby(
        ["date", "sym_root"],
        sort=False,
    ):
        g = g.sort_values("time_m").copy()
        keep = q4_keep_mask(
            g["midquote"].to_numpy(),
            half_window=q4_half_window,
            threshold=q4_threshold,
        )
        q4_removed += int((~keep).sum())
        pieces.append(g.loc[keep])

    q = (
        pd.concat(pieces, ignore_index=True)
        .sort_values(["date", "sym_root", "time_m"])
        .reset_index(drop=True)
    )
    stage_counts.append(("after_Q4_local_midquote_filter", len(q)))

    q["timestamp"] = (
        pd.to_datetime(q["date"])
        + pd.to_timedelta(q["time_m"], unit="s")
    )

    counts = pd.DataFrame(
        stage_counts,
        columns=["stage", "rows_remaining"],
    )
    counts["removed_from_previous_stage"] = (
        counts["rows_remaining"]
        .shift(1)
        .sub(counts["rows_remaining"])
        .fillna(0)
        .astype(int)
    )

    return q, counts


clean_quotes, cleaning_counts = clean_nbbo_quotes(nbbo_multi_raw)

display(cleaning_counts)


### What we had already learned before Q4

In the exploratory run, the working five-stock object had already applied the positive-price restrictions and Q2 (negative-spread removal). It contained:

- **2,139,715** observations entering Q3;
- **2,131,546** observations after Q3;
- **8,169** observations removed by the wide-spread filter;
- only about **0.382%** of the Q3 input discarded.

Despite removing a very small fraction of the data, the JPM observations around 09:45 returned to economically plausible levels near \$24.9.

This is precisely why a cleaning rule should not be judged by how many rows it deletes. A tiny number of extreme observations can exert disproportionate influence on a quadratic statistic such as realized covariance.

Barndorff-Nielsen et al. (2011) likewise report that several of their important price/quote/trade filters collectively remove less than one percent of observations in their application. Our percentage is **not directly comparable** because we use a different data representation and only a five-stock sample, but the qualitative lesson is the same: targeted filtering can matter greatly even when the number of deleted rows is small.


## 9. Re-synchronize after Q1–Q4 and compare the naive and cleaned panels

Cleaning must precede synchronization. If Q4 removes a bad quote immediately before a five-minute grid point, previous-tick sampling will now fall back to the most recent **valid** quote instead.

The diagnostics below compare the largest absolute five-minute return for each stock before and after cleaning. A useful cleaning pipeline should remove obvious data errors without mechanically suppressing legitimate market movement.


In [ ]:
prices_5min_clean = previous_tick_panel(
    clean_quotes,
    symbols,
    grid,
)

naive_returns = np.log(prices_5min_naive.astype(float)).diff()
clean_returns = np.log(prices_5min_clean.astype(float)).diff()

comparison = pd.DataFrame(
    {
        "naive_max_abs_5min_return": naive_returns.abs().max(),
        "clean_max_abs_5min_return": clean_returns.abs().max(),
    }
)

display(comparison)

print("\nCleaned five-minute prices:")
display(prices_5min_clean.head(10))

print("\nMissing values after cleaning and synchronization:")
display(prices_5min_clean.isna().sum())


In [ ]:
# Visual diagnostic: the JPM five-minute path before and after quote cleaning.
jpm_compare = pd.DataFrame(
    {
        "naive_midquote": prices_5min_naive["JPM"].astype(float),
        "clean_midquote": prices_5min_clean["JPM"].astype(float),
    }
)

ax = jpm_compare.plot(
    figsize=(11, 4),
    title="JPM five-minute NBBO midpoint: naive vs Q1–Q4 cleaned",
)
ax.set_xlabel("Time")
ax.set_ylabel("Midquote")
plt.show()


## 10. Why this matters for realized covariance

Suppose the synchronized return vector at intraday interval \(j\) is

$$
r_j =
\begin{bmatrix}
r_{1j}\\
\vdots\\
r_{Nj}
\end{bmatrix}.
$$

The standard realized covariance estimator is

$$
\widehat{\Sigma}_d
=
\sum_{j=1}^{M} r_jr_j'.
$$

This quadratic form means an erroneous return affects the measurement target nonlinearly:

- its variance contribution is \(r_{ij}^2\);
- every covariance involving asset \(i\) contains \(r_{ij}r_{kj}\).

A single artificial JPM return can therefore contaminate an entire row and column of the daily realized covariance matrix.

That is especially dangerous in this project because the realized covariance matrix is later treated as the **evaluation proxy** against which competing econometric and deep-learning forecasts are scored. If the proxy is corrupted at the measurement stage, a sophisticated evaluation protocol cannot repair it later.

The JPM example therefore establishes a design principle for `covharness`:

> **Market-data cleaning belongs inside the reproducible measurement layer and must be documented, tested and cited just like the covariance estimator itself.**

This notebook intentionally stops before constructing the final realized covariance matrix. The next production step is to freeze and unit-test the cleaning implementation, rerun the synchronized panel, and only then feed the returns into the existing realized-covariance code.


## References

Barndorff-Nielsen, O. E., Hansen, P. R., Lunde, A., & Shephard, N. (2009). *Realised kernels in practice: trades and quotes*. **The Econometrics Journal, 12**(3), C1–C32.

Barndorff-Nielsen, O. E., Hansen, P. R., Lunde, A., & Shephard, N. (2011). *Multivariate realised kernels: consistent positive semi-definite estimators of the covariation of equity prices with noise and non-synchronous trading*. **Journal of Econometrics, 162**(2), 149–169. https://doi.org/10.1016/j.jeconom.2010.07.009
